# T5 i inne rodziny transformerów

## Zadanie
Poznaj encoder–decoder T5 i porównaj go z GPT. Zobacz przepływ encoder self-attention, decoder causal attention, cross-attention oraz cel span corruption. Kod PyTorcha demonstruje tensorowe interfejsy, nie odtwarza pełnego T5.

## Rama teoretyczna: modele encoder–decoder i denoising

### Warunkowe generowanie sekwencji
W zadaniu seq2seq obserwujemy źródło $x=(x_1,\ldots,x_S)$ i chcemy modelować cel $y=(y_1,\ldots,y_T)$. Rozkład celu faktoryzuje się jako $p_\theta(y\mid x)=\prod_{t=1}^{T}p_\theta(y_t\mid y_{<t},x)$. Encoder zamienia całe źródło na reprezentacje $H_{enc}\in\mathbb R^{B\times S\times d}$. Decoder buduje kolejne reprezentacje celu, łącząc kontekst dotychczas wygenerowanych tokenów z kontekstem źródła.

### Trzy typy attention w encoder–decoder
Encoder self-attention jest dwukierunkowe: token źródłowy może korzystać z innych pozycji źródłowych, bo całe wejście jest znane. Decoder self-attention jest przyczynowe i maskuje przyszłe tokeny celu. Cross-attention używa $Q=H_{dec}W_Q$, $K=H_{enc}W_K$, $V=H_{enc}W_V$, dzięki czemu pozycje generowanego celu wybierają istotne fragmenty źródła. Attention maski pełnią różne funkcje: causal blokuje przyszłość, a padding mask blokuje sztuczne pozycje dodane do ujednolicenia długości batcha.

Teacher forcing przesuwa cel: decoder otrzymuje `BOS, y_1, ..., y_{T-1}`, a strata porównuje jego logity z `y_1, ..., y_T`. Tak jak w GPT obliczenia treningowe są równoległe po pozycjach, ale inference generuje tokeny krok po kroku.

### Text-to-text i span corruption
T5 traktuje wiele zadań jako mapowanie tekstu na tekst, np. prefiks `summarize:` lub `translate English to German:` instruuje model, jaki warunkowy cel ma wygenerować. W pretrainingu T5 losowe fragmenty wejścia (spany) zastępowane są sentinelami. Decoder odtwarza sentinel i treść usuniętego spanu. Zamiast przewidywać każdy oryginalny token jak klasyczny next-token LM, model uczy się rekonstrukcji wybranych fragmentów warunkowanej zachowanym kontekstem.

W uproszczonym przykładzie: źródło `a b c d e`, po usunięciu `c d` daje wejście `a b <extra_id_0> e`, a cel `<extra_id_0> c d <extra_id_1>`. Rzeczywisty preprocessing T5 obejmuje tokenizację SentencePiece, próbkowanie spanów, obsługę wielu sentinelów i szczegóły długości; komórka kodu poniżej pokazuje jedynie ideę transformacji.

### Związek architektury z zadaniem
Encoder-only (np. BERT) dobrze pasuje do zadań, w których potrzebna jest reprezentacja wejścia i predykcja etykiety lub spanów. Decoder-only (GPT) modeluje naturalnie kontynuację w jednym strumieniu tokenów. Encoder–decoder jawnie rozdziela źródło od autoregresyjnego celu, co jest wygodne dla tłumaczenia, streszczania i wielu transformacji. Można przystosować każdą rodzinę do wielu zadań; to rozróżnienie opisuje naturalny przepływ informacji, a nie bezwzględne ograniczenie.

Oryginalny Transformer z 2017 r. wprowadził encoder–decoder oparty na attention dla tłumaczenia. T5 rozwija tę konfigurację i text-to-text pretraining. GPT-3 jest dużym decoder-only LM, którego publikacja badała zachowanie few-shot. GPT-4 ma publiczny raport techniczny z ograniczonymi szczegółami implementacyjnymi. ChatGPT jest nazwą asystenta/systemu, a nie odrębnym typem bloku transformera.

### Zakres demonstracji
Kod notebooka sprawdza kształty i rolę cross-attention, ale nie zawiera token embeddingów, pozycyjnych biasów, padding masks, pełnego teacher-forcing loss, optymalizatora ani treningu seq2seq. To celowe: najpierw izolujemy przepływ informacji. Rozbudowa do trenowalnego T5 wymaga wszystkich tych elementów oraz wiernego preprocessingu i większego korpusu.

## 1. Encoder-only, decoder-only, encoder–decoder
- **BERT / encoder-only:** dwukierunkowa reprezentacja wejścia; klasyfikacja i ekstrakcja.
- **GPT / decoder-only:** causal self-attention; kontynuacja i generowanie.
- **T5 / encoder–decoder:** encoder czyta źródło, decoder generuje cel i pobiera kontekst przez cross-attention.

T5 ujmuje zadania jako text-to-text, np. `translate English to German: ...` albo `summarize: ...`. Wspólny format nie znaczy, że cel semantyczny zadań jest ten sam.

In [ ]:
import math,torch
from torch import nn
torch.manual_seed(23);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
class Attention(nn.Module):
    def __init__(self,d):
        super().__init__();self.q=nn.Linear(d,d);self.k=nn.Linear(d,d);self.v=nn.Linear(d,d);self.o=nn.Linear(d,d)
    def forward(self,qin,kvin,mask=None):
        q,k,v=self.q(qin),self.k(kvin),self.v(kvin);s=q@k.transpose(-2,-1)/math.sqrt(q.size(-1))
        if mask is not None:s=s.masked_fill(~mask,torch.finfo(s.dtype).min)
        return self.o(s.softmax(-1)@v)
class EncoderLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.a=Attention(d);self.n1=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d));self.n2=nn.LayerNorm(d)
    def forward(self,x):
        x=self.n1(x+self.a(x,x));return self.n2(x+self.ff(x))
class DecoderLayer(nn.Module):
    def __init__(self,d):
        super().__init__();self.sa=Attention(d);self.ca=Attention(d);self.n1=nn.LayerNorm(d);self.n2=nn.LayerNorm(d);self.ff=nn.Sequential(nn.Linear(d,4*d),nn.GELU(),nn.Linear(4*d,d));self.n3=nn.LayerNorm(d)
    def forward(self,y,mem):
        t=y.size(1);mask=torch.tril(torch.ones(t,t,dtype=torch.bool,device=y.device))
        y=self.n1(y+self.sa(y,y,mask));y=self.n2(y+self.ca(y,mem));return self.n3(y+self.ff(y))
enc=torch.randn(2,7,32,device=device);dec=torch.randn(2,4,32,device=device)
print("cross-attention:",Attention(32).to(device)(dec,enc).shape)
print("encoder:",EncoderLayer(32).to(device)(enc).shape)
print("decoder:",DecoderLayer(32).to(device)(dec,enc).shape)

## 2. Cross-attention i teacher forcing
Cross-attention używa zapytań z dekodera ($Q=H_{dec}W_Q$) oraz kluczy i wartości z encodera ($K=H_{enc}W_K$, $V=H_{enc}W_V$). Decoder dostaje cel przesunięty w prawo, by nie widzieć przyszłości. W pełnym treningu potrzebne są maski paddingowe dla źródła i celu.

## 3. Span corruption w T5
Pretraining T5 usuwa fragmenty (spany) wejścia i każe decoderowi odtworzyć je. Sentinel `<extra_id_0>` zastępuje usunięty span, a target zawiera sentinel i treść. To różni się od GPT, które przewiduje następny token całego tekstu. Przykład jest dydaktyczny i nie implementuje próbkowania spanów ani SentencePiece.

In [ ]:
def corrupt(tokens,start,length):
    removed=tokens[start:start+length];sent="<extra_id_0>"
    return tokens[:start]+[sent]+tokens[start+length:],[sent]+removed+["<extra_id_1>"]
s="bright acidity balances ripe citrus fruit".split();inp,target=corrupt(s,2,2)
print("oryginał:",s,"\nwejście encoder:",inp,"\ncel decoder:",target)

## 4. Porównanie rodzin

| Rodzina | Układ | Typowy cel / zastosowanie |
|---|---|---|
| Transformer (2017) | encoder + decoder | tłumaczenie; attention zastępuje rekurencję |
| GPT | decoder-only | next-token i generowanie |
| BERT | encoder-only | reprezentacje dwukierunkowe, pierwotnie masked LM |
| T5 | encoder–decoder | text-to-text, span corruption |
| GPT-3 | decoder-only, duża skala | few-shot/in-context learning opisane w publikacji |
| GPT-4 | szczegóły modelu w dużej mierze nieujawnione | model ogólnego przeznaczenia |
| ChatGPT | asystent dialogowy | system zoptymalizowany do interakcji |

## Podsumowanie ścieżki
Poznałeś Q/K/V, skalowanie, causal mask, multi-head attention i blok GPT; trenowałeś mini-GPT na recenzjach; omówiłeś skalę, GPT-3/GPT-4/ChatGPT; teraz porównałeś T5 i inne architektury. Dobieraj cel i architekturę do przepływu informacji, formatu zadania, kosztu i danych. Pełne treningi wymagają tokenizera, masek paddingowych, optymalizacji, ewaluacji, checkpointów i kontroli jakości danych.